# YouTube Short to Long Format Tracker
## Find the parent long-form video a short was cut from
1. Pick random short from local dataset
2. Fetch actual video metadata from YouTube (online)
3. Extract description & captions
4. Find links to parent long-form video

In [1]:
import pandas as pd
import numpy as np
import json
import random
import re
import subprocess
from pathlib import Path
import os

# Set random seed
random.seed(42)
np.random.seed(42)

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load YouTube videos metadata from local dataset
youtube_data_path = Path('../Data/YouTube/videos_metadata.csv')

if youtube_data_path.exists():
    df = pd.read_csv(youtube_data_path)
    print(f"✅ Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"Total videos: {len(df)}")
else:
    print(f"❌ File not found at {youtube_data_path}")

✅ Dataset loaded successfully!
Shape: (12839, 67)
Total videos: 12839


C:\Users\alire\AppData\Local\Temp\ipykernel_28120\3262750221.py:5: DtypeWarning: Columns (35,62,65) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(youtube_data_path)


In [3]:
# Step 1: Filter for shorts and pick a random one
print("\n" + "="*70)
print("STEP 1: SELECT RANDOM SHORT FROM LOCAL DATASET")
print("="*70)

if 'is_short' in df.columns:
    short_videos = df[df['is_short'] == True].copy()
    print(f"\nTotal shorts in dataset: {len(short_videos)}")
    
    if len(short_videos) > 0:
        random_short = short_videos.sample(n=1).iloc[0]
        video_id = random_short['videoId']
        
        print(f"\n🎬 Selected Short Video:")
        print(f"  Video ID: {video_id}")
        print(f"  Title: {random_short['title']}")
        print(f"  Channel: {random_short['channelTitle']}")
        print(f"  Duration: {random_short['duration']}")
        print(f"  YouTube URL: https://www.youtube.com/shorts/{video_id}")
    else:
        print(f"❌ No shorts found in dataset")
        video_id = None
else:
    print(f"❌ No 'is_short' column found")
    video_id = None


STEP 1: SELECT RANDOM SHORT FROM LOCAL DATASET

Total shorts in dataset: 4356

🎬 Selected Short Video:
  Video ID: a8YKVUCh_Zw
  Title: Bellissimiiii #perte #unboxing #blindbox #sushicat
  Channel: Smibie Channel
  Duration: PT58S
  YouTube URL: https://www.youtube.com/shorts/a8YKVUCh_Zw


In [4]:
# Step 2: Fetch the actual video metadata from YouTube using yt-dlp
print("\n" + "="*70)
print("STEP 2: FETCH ACTUAL VIDEO METADATA FROM YOUTUBE (ONLINE)")
print("="*70)

if video_id:
    youtube_url = f"https://www.youtube.com/shorts/{video_id}"
    output_dir = Path('./youtube_metadata')
    output_dir.mkdir(exist_ok=True)
    
    print(f"\n🌐 Fetching metadata for: {youtube_url}")
    print(f"This may take a moment...\n")
    
    try:
        # Use yt-dlp to extract info
        cmd = [
            'yt-dlp',
            '--dump-json',
            '--write-info-json',
            '--skip-download',
            '-o', str(output_dir / '%(id)s'),
            '--write-sub',
            '--sub-format', 'json3',
            youtube_url
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        
        if result.returncode == 0:
            print(f"✅ Successfully fetched video metadata!")
            
            # Parse the JSON output
            video_data = json.loads(result.stdout)
            
            # Save full metadata
            json_path = output_dir / f"{video_id}_info.json"
            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(video_data, f, indent=2, ensure_ascii=False)
            
            print(f"✅ Metadata saved to: {json_path}")
        else:
            print(f"❌ Error fetching metadata:")
            print(result.stderr)
            video_data = None
            
    except subprocess.TimeoutExpired:
        print(f"❌ Timeout while fetching video (taking too long)")
        video_data = None
    except FileNotFoundError:
        print(f"❌ yt-dlp not found. Install it with: pip install yt-dlp")
        video_data = None
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        video_data = None
else:
    print(f"❌ No video ID available")
    video_data = None


STEP 2: FETCH ACTUAL VIDEO METADATA FROM YOUTUBE (ONLINE)

🌐 Fetching metadata for: https://www.youtube.com/shorts/a8YKVUCh_Zw
This may take a moment...

✅ Successfully fetched video metadata!
✅ Metadata saved to: youtube_metadata\a8YKVUCh_Zw_info.json


In [5]:
# Step 3: Extract description and look for links
print("\n" + "="*70)
print("STEP 3: EXTRACT DESCRIPTION & FIND LINKS")
print("="*70)

if video_data:
    description = video_data.get('description', '')
    
    print(f"\n📝 FULL DESCRIPTION FROM YOUTUBE:")
    print("-" * 70)
    if description:
        print(description)
    else:
        print("[No description]")
    
    # Extract all YouTube links
    def extract_youtube_links(text):
        """Extract YouTube video IDs and links from text"""
        if not text:
            return []
        
        links = []
        
        # Pattern 1: youtube.com/watch?v=VIDEO_ID
        pattern1 = r'(?:https?://)?(?:www\.)?youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})'
        # Pattern 2: youtu.be/VIDEO_ID
        pattern2 = r'(?:https?://)?youtu\.be/([a-zA-Z0-9_-]{11})'
        # Pattern 3: youtube.com/shorts/VIDEO_ID
        pattern3 = r'(?:https?://)?(?:www\.)?youtube\.com/shorts/([a-zA-Z0-9_-]{11})'
        
        matches1 = re.finditer(pattern1, text)
        matches2 = re.finditer(pattern2, text)
        matches3 = re.finditer(pattern3, text)
        
        for match in matches1:
            vid_id = match.group(1)
            full_url = match.group(0)
            links.append({'id': vid_id, 'url': full_url, 'type': 'watch'})
        
        for match in matches2:
            vid_id = match.group(1)
            full_url = match.group(0)
            links.append({'id': vid_id, 'url': full_url, 'type': 'youtu.be'})
        
        for match in matches3:
            vid_id = match.group(1)
            full_url = match.group(0)
            links.append({'id': vid_id, 'url': full_url, 'type': 'shorts'})
        
        return links
    
    found_links = extract_youtube_links(description)
    
    print(f"\n🔗 YOUTUBE LINKS FOUND IN DESCRIPTION:")
    print("-" * 70)
    if found_links:
        for i, link in enumerate(found_links, 1):
            print(f"\n{i}. Video ID: {link['id']}")
            print(f"   Type: {link['type']}")
            print(f"   Full URL: {link['url']}")
            print(f"   Direct URL: https://www.youtube.com/watch?v={link['id']}")
    else:
        print("[No YouTube links found in description]")
else:
    print(f"❌ No video data available")


STEP 3: EXTRACT DESCRIPTION & FIND LINKS

📝 FULL DESCRIPTION FROM YOUTUBE:
----------------------------------------------------------------------
[No description]

🔗 YOUTUBE LINKS FOUND IN DESCRIPTION:
----------------------------------------------------------------------
[No YouTube links found in description]


In [6]:
# Step 4: Extract and search captions
print("\n" + "="*70)
print("STEP 4: EXTRACT CAPTIONS & SEARCH FOR LINKS")
print("="*70)

if video_data:
    # Check if subtitles were downloaded
    subtitles = video_data.get('subtitles', {})
    auto_captions = video_data.get('automatic_captions', {})
    
    all_captions = {**subtitles, **auto_captions}
    
    if all_captions:
        print(f"\nAvailable caption languages: {list(all_captions.keys())}")
        
        caption_text = ""
        # Try to get captions in order of preference
        for lang in ['en', 'it', 'fr', 'es', 'de']:
            if lang in all_captions:
                caption_urls = all_captions[lang]
                if caption_urls:
                    print(f"\n✅ Found captions in '{lang}'")
                    # Captions are provided as URLs, we could fetch them
                    # For now, just note that they're available
                    break
    else:
        print(f"\n[No captions available for this video]")
else:
    print(f"❌ No video data available")


STEP 4: EXTRACT CAPTIONS & SEARCH FOR LINKS

Available caption languages: ['ab', 'aa', 'af', 'ak', 'sq', 'am', 'ar', 'hy', 'as', 'ay', 'az', 'bn', 'ba', 'eu', 'be', 'bho', 'bs', 'br', 'bg', 'my', 'ca', 'ceb', 'zh-Hans', 'zh-Hant', 'co', 'hr', 'cs', 'da', 'dv', 'nl', 'dz', 'en', 'eo', 'et', 'ee', 'fo', 'fj', 'fil', 'fi', 'fr', 'gaa', 'gl', 'lg', 'ka', 'de', 'el', 'gn', 'gu', 'ht', 'ha', 'haw', 'iw', 'hi', 'hmn', 'hu', 'is', 'ig', 'id', 'iu', 'ga', 'it-orig', 'it', 'ja', 'jv', 'kl', 'kn', 'kk', 'kha', 'km', 'rw', 'ko', 'kri', 'ku', 'ky', 'lo', 'la', 'lv', 'ln', 'lt', 'lua', 'luo', 'lb', 'mk', 'mg', 'ms', 'ml', 'mt', 'gv', 'mi', 'mr', 'mn', 'mfe', 'ne', 'new', 'nso', 'no', 'ny', 'oc', 'or', 'om', 'os', 'pam', 'ps', 'fa', 'pl', 'pt', 'pt-PT', 'pa', 'qu', 'ro', 'rn', 'ru', 'sm', 'sg', 'sa', 'gd', 'sr', 'crs', 'sn', 'sd', 'si', 'sk', 'sl', 'so', 'st', 'es', 'su', 'sw', 'ss', 'sv', 'tg', 'ta', 'tt', 'te', 'th', 'bo', 'ti', 'to', 'ts', 'tn', 'tum', 'tr', 'tk', 'uk', 'ur', 'ug', 'uz', 've', 'v

In [7]:
# Step 5: Display full video JSON
print("\n" + "="*70)
print("STEP 5: COMPLETE VIDEO METADATA (JSON)")
print("="*70)

if video_data:
    print(f"\n📋 Full metadata saved. Showing key fields:")
    key_fields = {
        'id': video_data.get('id'),
        'title': video_data.get('title'),
        'channel': video_data.get('channel'),
        'duration': video_data.get('duration'),
        'description': video_data.get('description', '')[:200] + '...' if video_data.get('description') else 'N/A',
        'upload_date': video_data.get('upload_date'),
        'view_count': video_data.get('view_count'),
        'like_count': video_data.get('like_count'),
        'comment_count': video_data.get('comment_count'),
        'url': video_data.get('webpage_url'),
        'has_subtitles': len(video_data.get('subtitles', {})) > 0,
        'has_automatic_captions': len(video_data.get('automatic_captions', {})) > 0
    }
    
    print(json.dumps(key_fields, indent=2, ensure_ascii=False, default=str))
    
    print(f"\n\n🔍 FULL JSON available at: {output_dir / f'{video_id}_info.json'}")
else:
    print(f"❌ No video data available")


STEP 5: COMPLETE VIDEO METADATA (JSON)

📋 Full metadata saved. Showing key fields:
{
  "id": "a8YKVUCh_Zw",
  "title": "Bellissimiiii #perte #unboxing #blindbox #sushicat",
  "channel": "Smibie Channel",
  "duration": 58,
  "description": "N/A",
  "upload_date": "20241031",
  "view_count": 82758,
  "like_count": 3813,
  "comment_count": 13,
  "url": "https://www.youtube.com/watch?v=a8YKVUCh_Zw",
  "has_subtitles": false,
  "has_automatic_captions": true
}


🔍 FULL JSON available at: youtube_metadata\a8YKVUCh_Zw_info.json


In [8]:
# Step 6: Summary and recommendations
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

if video_data:
    print(f"\n✅ Successfully retrieved online metadata for short video: {video_id}")
    print(f"\n📊 Analysis Results:")
    
    if found_links:
        print(f"  ✅ Found {len(found_links)} YouTube link(s) in description")
        print(f"  These could be the parent long-form video(s)")
        for link in found_links:
            print(f"     - {link['id']}")
    else:
        print(f"  ❌ No YouTube links found in description")
        print(f"  The short may not explicitly link to the parent video")
    
    print(f"\n📁 Output files:")
    print(f"  - {output_dir / f'{video_id}_info.json'} (Full metadata)")
else:
    print(f"❌ Could not retrieve video metadata")


SUMMARY

✅ Successfully retrieved online metadata for short video: a8YKVUCh_Zw

📊 Analysis Results:
  ❌ No YouTube links found in description
  The short may not explicitly link to the parent video

📁 Output files:
  - youtube_metadata\a8YKVUCh_Zw_info.json (Full metadata)
